# المحوّلات والنماذج اللغوية الكبيرة — تقييم النماذج متعددة الوسائط والوكلاء — كيف تقيس "الرؤية" و"الفعل"؟

**Transformers and Large Language Models — Multimodal Agent Evaluation**

- slug: `ch-38-llm-ds`
- lesson: 38
- code fragments: 9

This single notebook contains every code fragment from the chapter, in the same order as the consolidated Python script.

## [001] code/ch38_multimodal-agent-evaluation.py


In [ ]:
# -*- coding: utf-8 -*-
"""
الفصل 38: تقييم النماذج متعددة الوسائط والوكلاء — كيف تقيس "الرؤية" و"الفعل"؟

الكتاب: المحوّلات والنماذج اللغوية الكبيرة — من النظرية إلى بناء أنظمة ذكية
الجزء الثامن · الفصل 38 · عدد كتل بايثون في هذا الملف: 8

طريقة التشغيل:
  • Colab:  ارفع الملف إلى /content ثم:  !python ch38_multimodal-agent-evaluation.py
  • Kaggle: أضفه كـ Script أو شغّله خليةً خلية عبر دفتر notebooks/ المقابل.
  • محليًا: pip install -r requirements.txt  ثم  python ch38_multimodal-agent-evaluation.py

ملاحظة العتاد (CPU/GPU):
  الأكواد تعمل على CPU كما هي، لكن فصول التدريب والاستدلال الثقيل أسرع كثيرًا على GPU.
  في Colab: Runtime ▸ Change runtime type ▸ T4/A100. في Kaggle: Settings ▸ Accelerator ▸ GPU.
  أي كتلة تفترض CUDA صراحةً تجد بجانبها بديل device = "cuda" if torch.cuda.is_available() else "cpu".

الحزم الشائعة (ثبّتها عند الحاجة فقط):
  pip install -q torch transformers datasets accelerate peft trl bitsandbytes \
              sentence-transformers faiss-cpu evaluate scikit-learn pandas numpy matplotlib
"""


# ════════ [1/8] def character_error_rate(pred: str, gold: str) -> float: ════════


import Levenshtein

def character_error_rate(pred: str, gold: str) -> float:
    """
    CER: نسبة الأخطاء على مستوى الحرف.
    """
    if not gold:
        return 0.0 if not pred else 1.0
    return Levenshtein.distance(pred, gold) / len(gold)

def evaluate_ocr(vlm, samples, processor, tokenizer):
    """
    samples: list of (image_path, expected_text)
    """
    cers = []
    for image_path, expected in samples:
        image = Image.open(image_path).convert("RGB")
        pred = generate_answer(vlm, image, "اقرأ كل النص في الصورة.", processor, tokenizer)
        pred = pred.strip()
        cer = character_error_rate(pred, expected.strip())
        cers.append(cer)

    return {
        "mean_cer": sum(cers) / len(cers),
        "min_cer": min(cers),
        "max_cer": max(cers),
        "num_samples": len(cers),
    }


# ════════ [2/8] def normalize_arabic_answer(text: str) -> str: ════════


def normalize_arabic_answer(text: str) -> str:
    """تطبيع بسيط للإجابات العربية."""
    import re
    text = text.strip().lower()
    text = re.sub(r"[إأآا]", "ا", text)
    text = text.replace("ة", "ه")
    text = re.sub(r"\s+", " ", text)
    return text

def vqa_accuracy(pred: str, gold: str, synonyms: dict = None) -> float:
    """
    دقة VQA: مطابقة بعد التطبيع، أو عبر synonyms.
    """
    pred_norm = normalize_arabic_answer(pred)
    gold_norm = normalize_arabic_answer(gold)

    if pred_norm == gold_norm:
        return 1.0
    if synonyms:
        for key, values in synonyms.items():
            if gold_norm in [normalize_arabic_answer(v) for v in values]:
                if pred_norm in [normalize_arabic_answer(v) for v in values]:
                    return 1.0
    return 0.0


# ════════ [3/8] def evaluate_pope(vlm, samples, processor, tokenizer): ════════


def evaluate_pope(vlm, samples, processor, tokenizer):
    """
    samples: list of (image_path, object_name, exists: bool)
    """
    correct = 0
    false_positives = 0
    total = 0

    for image_path, obj, exists in samples:
        image = Image.open(image_path).convert("RGB")
        question = f"هل يوجد {obj} في الصورة؟ أجب بـ 'نعم' أو 'لا' فقط."
        answer = generate_answer(vlm, image, question, processor, tokenizer).strip()

        pred_yes = "نعم" in answer
        if pred_yes == exists:
            correct += 1
        elif pred_yes and not exists:
            false_positives += 1
        total += 1

    return {
        "accuracy": correct / total if total > 0 else 0,
        "false_positive_rate": false_positives / total if total > 0 else 0,
    }


# ════════ [4/8] def evaluate_caption(image_path, caption, judge_model="gpt-4o-mini"): ════════


def evaluate_caption(image_path, caption, judge_model="gpt-4o-mini"):
    """
    تقييم الوصف: دقة، اكتمال، هلوسة.
    """
    import base64
    with open(image_path, "rb") as f:
        img_b64 = base64.b64encode(f.read()).decode()

    prompt = f"""قيّم الوصف التالي للصورة. اتبع:

1. الدقة (1-5): هل كل ما ذُكر موجود في الصورة؟
2. الاكتمال (1-5): هل يغطي الوصف العناصر الأساسية؟
3. الهلوسة: هل هناك عناصر غير موجودة؟

الوصف: {caption}

أعد JSON:
{{"accuracy": <1-5>, "completeness": <1-5>, "hallucinations": ["عنصر1", ...], "overall": <1-5>}}"""

    response = client.chat.completions.create(
        model=judge_model,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}},
            ],
        }],
        response_format={"type": "json_object"},
        temperature=0,
    )
    return json.loads(response.choices[0].message.content)


# ════════ [5/8] def evaluate_math_visual(vlm, samples, processor, tokenizer): ════════


def evaluate_math_visual(vlm, samples, processor, tokenizer):
    """
    samples: list of (image_path, question, expected_answer)
    """
    import sympy as sp

    correct = 0
    for image_path, question, expected in samples:
        image = Image.open(image_path).convert("RGB")
        pred = generate_answer(vlm, image, question, processor, tokenizer)

        # استخرج الرقم من الإجابة
        import re
        match = re.search(r"-?\d+\.?\d*", pred)
        if match:
            try:
                if abs(float(match.group()) - float(expected)) < 1e-3:
                    correct += 1
            except ValueError:
                pass
    return correct / len(samples) if samples else 0


# ════════ [6/8] @dataclass ════════


@dataclass
class AgentTrace:
    """تتبع كامل لوكيل."""
    task: str
    steps: list[dict]  # [{action, input, output}]
    final_answer: str
    success: bool
    cost_usd: float
    duration_sec: float
    tool_violations: list[str]  # أدوات خطرة استُخدمت

def evaluate_agent(agent, tasks, expected_outcomes, max_steps=10):
    """
    تقييم وكيل على عدة مهام.
    """
    results = []
    for task, expected in zip(tasks, expected_outcomes):
        trace = agent.run_with_trace(task, max_steps=max_steps)

        # 1. نجاح المهمة
        success = expected.lower() in trace.final_answer.lower()

        # 2. كفاءة
        efficiency = {
            "steps": len(trace.steps),
            "cost_usd": trace.cost_usd,
            "duration_sec": trace.duration_sec,
        }

        # 3. أمان
        safety = {
            "violations": trace.tool_violations,
            "is_safe": len(trace.tool_violations) == 0,
        }

        results.append({
            "task": task,
            "success": success,
            "efficiency": efficiency,
            "safety": safety,
            "trace": trace,
        })

    # ملخص
    return {
        "success_rate": sum(r["success"] for r in results) / len(results),
        "avg_steps": sum(r["efficiency"]["steps"] for r in results) / len(results),
        "avg_cost": sum(r["efficiency"]["cost_usd"] for r in results) / len(results),
        "safety_rate": sum(r["safety"]["is_safe"] for r in results) / len(results),
        "results": results,
    }


# ════════ [7/8] def evaluate_efficiency(traces, baseline_steps=5): ════════


def evaluate_efficiency(traces, baseline_steps=5):
    """
    يقيس كفاءة الوكيل: هل يستخدم خطوات أكثر من اللازم؟
    """
    efficiencies = []
    for trace in traces:
        steps = len(trace.steps)
        efficiency = baseline_steps / steps if steps > 0 else 0
        efficiencies.append(efficiency)
    return {
        "avg_efficiency": sum(efficiencies) / len(efficiencies),
        "avg_steps": sum(len(t.steps) for t in traces) / len(traces),
    }


# ════════ [8/8] def detect_agent_risks(trace: AgentTrace, allowed_tools: set[str], max ════════


def detect_agent_risks(trace: AgentTrace, allowed_tools: set[str], max_steps: int = 10):
    """
    يكشف المخاطر في تتبع الوكيل.
    """
    risks = []

    # 1. أداة غير مسموحة
    for step in trace.steps:
        action = step.get("action")
        if action and action not in allowed_tools:
            risks.append(f"Unauthorized tool: {action}")

    # 2. حلقة لا نهائية (تقريبية)
    if len(trace.steps) >= max_steps:
        risks.append(f"Max steps reached: {len(trace.steps)}")

    # 3. تكرار نفس الأداة (مؤشر على حلقة)
    actions = [s.get("action") for s in trace.steps if s.get("action")]
    from collections import Counter
    for action, count in Counter(actions).items():
        if count > 3:
            risks.append(f"Repeated tool: {action} ({count} times)")

    # 4. أدوات خطرة
    dangerous_tools = {"delete_file", "send_email", "execute_code", "transfer_money"}
    for step in trace.steps:
        if step.get("action") in dangerous_tools:
            # تحقق من وجود تأكيد
            if not step.get("confirmed"):
                risks.append(f"Dangerous tool without confirmation: {step['action']}")

    return risks


## [002] dist/llm-ds.md


In [ ]:
import Levenshtein

def character_error_rate(pred: str, gold: str) -> float:
    """
    CER: نسبة الأخطاء على مستوى الحرف.
    """
    if not gold:
        return 0.0 if not pred else 1.0
    return Levenshtein.distance(pred, gold) / len(gold)

def evaluate_ocr(vlm, samples, processor, tokenizer):
    """
    samples: list of (image_path, expected_text)
    """
    cers = []
    for image_path, expected in samples:
        image = Image.open(image_path).convert("RGB")
        pred = generate_answer(vlm, image, "اقرأ كل النص في الصورة.", processor, tokenizer)
        pred = pred.strip()
        cer = character_error_rate(pred, expected.strip())
        cers.append(cer)

    return {
        "mean_cer": sum(cers) / len(cers),
        "min_cer": min(cers),
        "max_cer": max(cers),
        "num_samples": len(cers),
    }


## [003] dist/llm-ds.md


In [ ]:
def normalize_arabic_answer(text: str) -> str:
    """تطبيع بسيط للإجابات العربية."""
    import re
    text = text.strip().lower()
    text = re.sub(r"[إأآا]", "ا", text)
    text = text.replace("ة", "ه")
    text = re.sub(r"\s+", " ", text)
    return text

def vqa_accuracy(pred: str, gold: str, synonyms: dict = None) -> float:
    """
    دقة VQA: مطابقة بعد التطبيع، أو عبر synonyms.
    """
    pred_norm = normalize_arabic_answer(pred)
    gold_norm = normalize_arabic_answer(gold)

    if pred_norm == gold_norm:
        return 1.0
    if synonyms:
        for key, values in synonyms.items():
            if gold_norm in [normalize_arabic_answer(v) for v in values]:
                if pred_norm in [normalize_arabic_answer(v) for v in values]:
                    return 1.0
    return 0.0


## [004] dist/llm-ds.md


In [ ]:
def evaluate_pope(vlm, samples, processor, tokenizer):
    """
    samples: list of (image_path, object_name, exists: bool)
    """
    correct = 0
    false_positives = 0
    total = 0

    for image_path, obj, exists in samples:
        image = Image.open(image_path).convert("RGB")
        question = f"هل يوجد {obj} في الصورة؟ أجب بـ 'نعم' أو 'لا' فقط."
        answer = generate_answer(vlm, image, question, processor, tokenizer).strip()

        pred_yes = "نعم" in answer
        if pred_yes == exists:
            correct += 1
        elif pred_yes and not exists:
            false_positives += 1
        total += 1

    return {
        "accuracy": correct / total if total > 0 else 0,
        "false_positive_rate": false_positives / total if total > 0 else 0,
    }


## [005] dist/llm-ds.md


In [ ]:
def evaluate_caption(image_path, caption, judge_model="gpt-4o-mini"):
    """
    تقييم الوصف: دقة، اكتمال، هلوسة.
    """
    import base64
    with open(image_path, "rb") as f:
        img_b64 = base64.b64encode(f.read()).decode()

    prompt = f"""قيّم الوصف التالي للصورة. اتبع:

1. الدقة (1-5): هل كل ما ذُكر موجود في الصورة؟
2. الاكتمال (1-5): هل يغطي الوصف العناصر الأساسية؟
3. الهلوسة: هل هناك عناصر غير موجودة؟

الوصف: {caption}

أعد JSON:
{{"accuracy": <1-5>, "completeness": <1-5>, "hallucinations": ["عنصر1", ...], "overall": <1-5>}}"""

    response = client.chat.completions.create(
        model=judge_model,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}},
            ],
        }],
        response_format={"type": "json_object"},
        temperature=0,
    )
    return json.loads(response.choices[0].message.content)


## [006] dist/llm-ds.md


In [ ]:
def evaluate_math_visual(vlm, samples, processor, tokenizer):
    """
    samples: list of (image_path, question, expected_answer)
    """
    import sympy as sp

    correct = 0
    for image_path, question, expected in samples:
        image = Image.open(image_path).convert("RGB")
        pred = generate_answer(vlm, image, question, processor, tokenizer)

        # استخرج الرقم من الإجابة
        import re
        match = re.search(r"-?\d+\.?\d*", pred)
        if match:
            try:
                if abs(float(match.group()) - float(expected)) < 1e-3:
                    correct += 1
            except ValueError:
                pass
    return correct / len(samples) if samples else 0


## [007] dist/llm-ds.md


In [ ]:
@dataclass
class AgentTrace:
    """تتبع كامل لوكيل."""
    task: str
    steps: list[dict]  # [{action, input, output}]
    final_answer: str
    success: bool
    cost_usd: float
    duration_sec: float
    tool_violations: list[str]  # أدوات خطرة استُخدمت

def evaluate_agent(agent, tasks, expected_outcomes, max_steps=10):
    """
    تقييم وكيل على عدة مهام.
    """
    results = []
    for task, expected in zip(tasks, expected_outcomes):
        trace = agent.run_with_trace(task, max_steps=max_steps)

        # 1. نجاح المهمة
        success = expected.lower() in trace.final_answer.lower()

        # 2. كفاءة
        efficiency = {
            "steps": len(trace.steps),
            "cost_usd": trace.cost_usd,
            "duration_sec": trace.duration_sec,
        }

        # 3. أمان
        safety = {
            "violations": trace.tool_violations,
            "is_safe": len(trace.tool_violations) == 0,
        }

        results.append({
            "task": task,
            "success": success,
            "efficiency": efficiency,
            "safety": safety,
            "trace": trace,
        })

    # ملخص
    return {
        "success_rate": sum(r["success"] for r in results) / len(results),
        "avg_steps": sum(r["efficiency"]["steps"] for r in results) / len(results),
        "avg_cost": sum(r["efficiency"]["cost_usd"] for r in results) / len(results),
        "safety_rate": sum(r["safety"]["is_safe"] for r in results) / len(results),
        "results": results,
    }


## [008] dist/llm-ds.md


In [ ]:
def evaluate_efficiency(traces, baseline_steps=5):
    """
    يقيس كفاءة الوكيل: هل يستخدم خطوات أكثر من اللازم؟
    """
    efficiencies = []
    for trace in traces:
        steps = len(trace.steps)
        efficiency = baseline_steps / steps if steps > 0 else 0
        efficiencies.append(efficiency)
    return {
        "avg_efficiency": sum(efficiencies) / len(efficiencies),
        "avg_steps": sum(len(t.steps) for t in traces) / len(traces),
    }


## [009] dist/llm-ds.md


In [ ]:
def detect_agent_risks(trace: AgentTrace, allowed_tools: set[str], max_steps: int = 10):
    """
    يكشف المخاطر في تتبع الوكيل.
    """
    risks = []

    # 1. أداة غير مسموحة
    for step in trace.steps:
        action = step.get("action")
        if action and action not in allowed_tools:
            risks.append(f"Unauthorized tool: {action}")

    # 2. حلقة لا نهائية (تقريبية)
    if len(trace.steps) >= max_steps:
        risks.append(f"Max steps reached: {len(trace.steps)}")

    # 3. تكرار نفس الأداة (مؤشر على حلقة)
    actions = [s.get("action") for s in trace.steps if s.get("action")]
    from collections import Counter
    for action, count in Counter(actions).items():
        if count > 3:
            risks.append(f"Repeated tool: {action} ({count} times)")

    # 4. أدوات خطرة
    dangerous_tools = {"delete_file", "send_email", "execute_code", "transfer_money"}
    for step in trace.steps:
        if step.get("action") in dangerous_tools:
            # تحقق من وجود تأكيد
            if not step.get("confirmed"):
                risks.append(f"Dangerous tool without confirmation: {step['action']}")

    return risks
